In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [11]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

# Kaggle competition notebooks expose the CSV bundle under /kaggle/input.
# Public uploads can also land under /kaggle/input/datasets/<owner>/<dataset>/...
# such as the sample upload path you gave: /kaggle/input/datasets/ravioshankar/sample-for-test
COMP = None
candidate_roots = [
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets/ravioshankar/sample-for-test'),
    Path('/mnt/data'),
    Path('/content'),
    Path.cwd(),
]

for root in candidate_roots:
    if not root.exists():
        continue

    # First, prefer the exact competition-style root if it already contains all three files.
    if all((root / name).exists() for name in ['train.csv', 'test.csv', 'sample_submission.csv']):
        COMP = root
        break

    # Otherwise, walk the tree and accept any folder that contains the minimal file set.
    for csv_name in ['train.csv', 'test.csv', 'sample_submission.csv']:
        matches = sorted(root.rglob(csv_name))
        for csv_path in matches:
            folder = csv_path.parent
            if any((folder / name).exists() for name in ['train.csv', 'test.csv', 'sample_submission.csv']):
                COMP = folder
                break
        if COMP is not None:
            break
    if COMP is not None:
        break

if COMP is None:
    raise FileNotFoundError(
        "Could not find the competition CSV inputs under the Kaggle input tree. "
        "The uploaded dataset path /kaggle/input/datasets/ravioshankar/sample-for-test "
        "was not found as a valid input bundle. "
        "Run this notebook in a Kaggle competition notebook or download the CSVs "
        "with the Kaggle API before executing Cell 2."
    )

TARGET_LABELS = [
    "Synovitis", "Baker's", "Contusion", "Fracture",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
]


In [12]:
# Cell 2 — train text baseline on expert-labeled rows (or load uploaded pickle)
train_csv = COMP / "train.csv"
if not train_csv.exists():
    print("No train.csv found in", COMP)
    print("This notebook upload only exposes a dataset upload path, so the text model cannot train from labels.")
    print("Expected Kaggle competition CSVs: train.csv, test.csv, sample_submission.csv")
    raise FileNotFoundError(f"Missing train.csv under {COMP}; this upload is insufficient for supervised training.")

train = pd.read_csv(train_csv)
label_cols = [c for c in TARGET_LABELS if c in train.columns]
mask = train[label_cols].notna().any(axis=1)
labeled = train.loc[mask].copy()
y = labeled[TARGET_LABELS].to_numpy(dtype=float)
y = np.nan_to_num(y, nan=0.0)

vectorizer = TfidfVectorizer(
    max_features=30_000,
    ngram_range=(1, 2),
    min_df=2,
    strip_accents="unicode",
    sublinear_tf=True,
)
X = vectorizer.fit_transform(labeled["Report"].fillna("").astype(str))

clf = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=42)
)
clf.fit(X, y)


,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](12,)","[ 0, 1, 2,..., 9,10,11]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,12
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,2090
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'


In [13]:
# Cell 3 — predict test (Report may be missing)
test = pd.read_csv(COMP / "test.csv")
sample = pd.read_csv(COMP / "sample_submission.csv")
if "Report" in test.columns:
    reports = test["Report"].fillna("").astype(str)
else:
    reports = pd.Series([""] * len(test), index=test.index)

Xt = vectorizer.transform(reports)
probas = np.column_stack([est.predict_proba(Xt)[:, 1] for est in clf.estimators_])

# If reports are empty, fall back to sample priors (0.5) so image models can replace later
if reports.str.strip().eq("").all():
    print("WARNING: all test reports empty — text model uninformative; use image/fusion.")
    probas = sample[TARGET_LABELS].to_numpy(dtype=float)

sub = sample.copy()
sub[TARGET_LABELS] = probas

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/home/robogo/workspace/siddha-lifestyle/kaggle/rsna-knee-abnormality-detection/submissions")
output_dir.mkdir(parents=True, exist_ok=True)
sub.to_csv(output_dir / "submission.csv", index=False)
print(sub.head())
print("Wrote submission.csv", len(sub), "to", output_dir / "submission.csv")

                                    StudyInstanceUID  ACL  MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.5  0.5   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.5  0.5   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.5  0.5   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA  PF OA  Effusion  \
0              0.5               0.5        0.5         0.5    0.5       0.5   
1              0.5               0.5        0.5         0.5    0.5       0.5   
2              0.5               0.5        0.5         0.5    0.5       0.5   

   Synovitis  Baker's  Contusion  Fracture  
0        0.5      0.5        0.5       0.5  
1        0.5      0.5        0.5       0.5  
2        0.5      0.5        0.5       0.5  
Wrote submission.csv 3 to /home/robogo/workspace/siddha-lifestyle/kaggle/rsna-knee-abnormality-detection/submissions/submission.csv
